# Bengaluru House Price Prediction
## Session 1: setup, load and first-pass cleaning

**Objective.** Load the raw Bengaluru house-price data, understand it, and reduce it to a clean frame that holds only the columns that can feed the model (`location`, `size`, `total_sqft`, `bath`, `price`) with no missing values.

**Target.** `price`, in **lakhs of INR** (1 lakh = 100,000 INR). Prices stay in lakhs throughout the project; nothing is converted here.

**Out of scope for this session.** Cleaning `total_sqft` (ranges and unit strings), encoding, scaling and the train/test split.

## 1. Setup and load

`df_raw` holds the data exactly as loaded and is never modified; every cleaning step works on a new frame. The findings written in prose describe the file whose SHA-256 begins `23ddf71be7d7` (printed when the file is loaded); a guard cell near the end warns if a run no longer matches.

In [1]:
import hashlib
import sys
from math import comb
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")

Python 3.13.18 | pandas 3.1.0 | numpy 2.5.4


In [2]:
# Kaggle's own file name is Bengaluru_House_Data.csv; the same data also circulates as
# bengaluru_house_prices.csv. Either name works.
CSV_NAMES = ["bengaluru_house_prices.csv", "Bengaluru_House_Data.csv"]

# The notebook lives in model/, so the data folder is one level up.
# "data/" also works when Jupyter is started from the repository root.
candidates = [folder / name for folder in (Path("../data"), Path("data")) for name in CSV_NAMES]
csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "No CSV found. Download 'Bengaluru House Price Data' from Kaggle and save it as "
        f"data/{CSV_NAMES[0]} (or data/{CSV_NAMES[1]}) in the repository root."
    )

csv_sha256 = hashlib.sha256(csv_path.read_bytes()).hexdigest()
df_raw = pd.read_csv(csv_path)
print(f"Loaded {csv_path} (sha256 {csv_sha256[:12]}...)")

Loaded ../data/bengaluru_house_prices.csv (sha256 23ddf71be7d7...)


## 2. Inspect the raw data

In [3]:
df_raw.shape

(13320, 9)

In [4]:
df_raw.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [5]:
df_raw.dtypes

area_type           str
availability        str
location            str
size                str
society             str
total_sqft          str
bath            float64
balcony         float64
price           float64
dtype: object

Text columns appear as `object` in pandas below 3 and as `str` from pandas 3 on; either way they are non-numeric. `total_sqft` is one of them: it holds ranges and unit strings, so it stays untouched until Session 2.

In [6]:
null_counts = df_raw.isnull().sum()
pd.DataFrame({"nulls": null_counts, "pct_missing": (null_counts / len(df_raw) * 100).round(2)})

,nulls,pct_missing
area_type,0,0.00
availability,0,0.00
location,1,0.01
size,16,0.12
society,5502,41.31
total_sqft,0,0.00
bath,73,0.55
balcony,609,4.57
price,0,0.00


In [7]:
df_raw["area_type"].value_counts()

area_type
Super built-up  Area    8790
Built-up  Area          2418
Plot  Area              2025
Carpet  Area              87
Name: count, dtype: int64

### Observed vs expected

The brief quotes approximate figures. The table puts them next to what the data contains.

In [8]:
expected = {  # figures quoted in the Session 1 brief (all approximate)
    "rows": 13000,
    "columns": 9,
    "society missing (%)": 40,
    "balcony missing": 600,
    "bath missing": 70,
    "size missing": 16,
    "location missing": 1,
}
observed = {
    "rows": len(df_raw),
    "columns": df_raw.shape[1],
    "society missing (%)": round(df_raw["society"].isnull().mean() * 100, 1),
    "balcony missing": int(df_raw["balcony"].isnull().sum()),
    "bath missing": int(df_raw["bath"].isnull().sum()),
    "size missing": int(df_raw["size"].isnull().sum()),
    "location missing": int(df_raw["location"].isnull().sum()),
}
pd.DataFrame({
    "expected (approx.)": pd.Series(expected, dtype=object),
    "observed": pd.Series(observed, dtype=object),
})

,expected (approx.),observed
rows,13000,13320
columns,9,9
society missing (%),40,41.3
balcony missing,600,609
bath missing,70,73
size missing,16,16
location missing,1,1


In [9]:
# Observation only: nothing is removed in this session.
n_repeats = int(df_raw.duplicated().sum())
print(f"Rows that exactly repeat an earlier row: {n_repeats:,} ({n_repeats / len(df_raw):.2%})")

Rows that exactly repeat an earlier row: 529 (3.97%)


### Findings

Every figure the brief expected holds: 13,320 rows and 9 columns; `society` is missing for 41.3% of the rows (5,502), `balcony` for 609, `bath` for 73, `size` for 16 and `location` for 1. `area_type`, `availability`, `total_sqft` and `price` have no nulls. `price` is numeric; `total_sqft` is text. 529 rows exactly repeat an earlier row; they are left in place for now.

## 3. Drop columns that cannot feed the model

Evidence for each decision, computed from the raw data:

In [10]:
DROP_COLS = ["area_type", "availability", "society", "balcony"]

evidence = pd.DataFrame({
    "dtype": df_raw[DROP_COLS].dtypes.astype(str),
    "pct_missing": (df_raw[DROP_COLS].isnull().mean() * 100).round(1),
    "n_unique": df_raw[DROP_COLS].nunique(),
    "top_values": pd.Series({c: df_raw[c].value_counts().head(3).to_dict() for c in DROP_COLS}),
})
with pd.option_context("display.max_colwidth", None):
    display(evidence)

,dtype,pct_missing,n_unique,top_values
area_type,str,0.0,4,"{'Super built-up Area': 8790, 'Built-up Area': 2418, 'Plot Area': 2025}"
availability,str,0.0,81,"{'Ready To Move': 10581, '18-Dec': 307, '18-May': 295}"
society,str,41.3,2688,"{'GrrvaGr': 80, 'PrarePa': 76, 'Sryalan': 59}"
balcony,float64,4.6,4,"{2.0: 5113, 1.0: 4897, 3.0: 1672}"


**Justification for each drop**

- **`area_type`**: it changes what `total_sqft` measures (super built-up, built-up, carpet or plot area), so it may well carry signal; it is left out of this first baseline as a judgment call, not because it is unusable, and it is logged in the README as a known limitation to test in Session 5.
- **`availability`**: it mixes the labels "Ready To Move" and "Immediate Possession" with possession-month strings such as "18-Dec" (81 distinct values in all), so it describes when a unit can be occupied rather than what the property is, and it cannot be used without parsing dates.
- **`society`**: 41.3% of its values are missing and the rest hold 2,688 distinct names, so it would be a sparse, high-cardinality categorical that cannot be encoded reliably.
- **`balcony`**: it is a minor amenity count with only four distinct values (0 to 3) and 609 missing entries, so dropping it avoids imputing values for otherwise complete rows and keeps the baseline on the core features.

In [11]:
decision_log = []


def log_step(decision, why, rows_before, rows_after):
    decision_log.append(
        {"decision": decision, "why": why, "rows_before": rows_before, "rows_after": rows_after}
    )


df = df_raw.drop(columns=DROP_COLS)
assert len(df) == len(df_raw)  # dropping columns never removes rows

log_step("Drop `area_type`", "Judgment call: changes what `total_sqft` measures; re-test in Session 5",
         len(df_raw), len(df))
log_step("Drop `availability`", "Possession label/month, not a property attribute", len(df_raw), len(df))
log_step("Drop `society`", "Large share missing and very high-cardinality", len(df_raw), len(df))
log_step("Drop `balcony`", "Minor amenity count; avoids imputing", len(df_raw), len(df))

df.shape

(13320, 5)

## 4. Remaining nulls in `location`, `size` and `bath`

Count them, look at where they sit, and check that they do not cluster before dropping them.

In [12]:
CORE_NULL_COLS = ["location", "size", "bath"]

nulls_now = df.isnull().sum()
display(nulls_now.to_frame("nulls"))
assert set(nulls_now[nulls_now > 0].index) <= set(CORE_NULL_COLS), "unexpected nulls outside location/size/bath"

,nulls
location,1
size,16
total_sqft,0
bath,73
price,0


In [13]:
null_flags = df[CORE_NULL_COLS].isnull()
null_mask = null_flags.any(axis=1)
n_null_rows = int(null_mask.sum())

print(f"Rows with at least one null in {CORE_NULL_COLS}: {n_null_rows} "
      f"({n_null_rows / len(df):.2%} of {len(df):,})")

# Which of the three columns are null together? (True = null)
null_flags[null_mask].value_counts().rename("rows").reset_index()

Rows with at least one null in ['location', 'size', 'bath']: 74 (0.56% of 13,320)


,location,size,bath,rows
0,False,False,True,57
1,False,True,True,16
2,True,False,False,1


In [14]:
# Raw rows (dropped columns still visible) for the null `location` and `size` cases.
df_raw[df["location"].isnull() | df["size"].isnull()]

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
568,Super built-up Area,Ready To Move,NaN,3 BHK,Grare S,1600,3.0,2.0,86.000
579,Plot Area,Immediate Possession,Sarjapur Road,NaN,Asiss B,1200 - 2400,NaN,NaN,34.185
1775,Plot Area,Immediate Possession,IVC Road,NaN,Orana N,2000 - 5634,NaN,NaN,124.000
2264,Plot Area,Immediate Possession,Banashankari,NaN,NaN,2400,NaN,NaN,460.000
2809,Plot Area,Immediate Possession,Sarjapur Road,NaN,AsdiaAr,1200 - 2400,NaN,NaN,28.785
2862,Plot Area,Immediate Possession,Devanahalli,NaN,Ajleyor,1500 - 2400,NaN,NaN,46.800
5333,Plot Area,Immediate Possession,Devanahalli,NaN,Emngs S,2100 - 5405,NaN,NaN,177.115
6423,Plot Area,Immediate Possession,Whitefield,NaN,SRniaGa,2324,NaN,NaN,26.730
6636,Plot Area,Immediate Possession,Jigani,NaN,S2enste,1500,NaN,NaN,25.490
6719,Plot Area,Immediate Possession,Hoskote,NaN,SJowsn,800 - 2660,NaN,NaN,28.545


### What the null rows have in common

A read-only look: `total_sqft` is not modified here (cleaning it is Session 2); the cell only tests whether each value is written as a range such as "2100 - 2850".

In [15]:
is_range = df["total_sqft"].str.contains(r"\d\s*-\s*\d")

groups = {
    "all rows": pd.Series(True, index=df.index),
    "any null row": null_mask,
    "bath only": null_flags["bath"] & ~null_flags["size"] & ~null_flags["location"],
    "size and bath": null_flags["size"],
    "location only": null_flags["location"],
}
rows = []
for name, m in groups.items():
    rows.append({
        "group": name,
        "rows": int(m.sum()),
        "total_sqft is a range": int(is_range[m].sum()),
        "% range": round(is_range[m].mean() * 100, 1),
        "% balcony also missing": round(df_raw.loc[m, "balcony"].isnull().mean() * 100, 1),
        "% Plot Area": round(df_raw.loc[m, "area_type"].str.contains("Plot").mean() * 100, 1),
    })
pd.DataFrame(rows).set_index("group")

,rows,total_sqft is a range,% range,% balcony also missing,% Plot Area
group,,,,,
all rows,13320,201,1.5,4.6,15.2
any null row,74,57,77.0,98.6,21.6
bath only,57,49,86.0,100.0,0.0
size and bath,16,8,50.0,100.0,100.0
location only,1,0,0.0,0.0,0.0


### Do the null rows cluster?

By `location` first, which is the check the brief asks for. Two questions, each judged against random scatter (the same number of null rows placed at random over all listings, 2,000 times): does one location dominate, and is any location over-represented for its size? Then by `area_type` (read from the raw frame, because the column is already gone from `df`) and by position in the file.

In [16]:
def tail_prob(k, n, p):
    """P(X >= k) for X ~ Binomial(n, p): the chance that a location holding a share p of all listings
    receives k or more of n null rows scattered at random. Small = more null rows than its size explains."""
    return sum(comb(n, i) * p**i * (1 - p) ** (n - i) for i in range(k, n + 1))


loc_all = df["location"].value_counts()                    # listings per location, all rows
loc_null = df.loc[null_mask, "location"].value_counts()    # null rows per location (null locations excluded)
n_known_null = int(loc_null.sum())

by_location = pd.DataFrame({"null_rows": loc_null, "all_rows": loc_all.reindex(loc_null.index)})
by_location["null_rate_%"] = (by_location["null_rows"] / by_location["all_rows"] * 100).round(1)
# Null rows a location would get if the nulls were scattered at random over all listings.
exp_loc = n_known_null * by_location["all_rows"] / loc_all.sum()
by_location["expected_if_random"] = exp_loc.round(1)
by_location["tail_prob"] = [
    tail_prob(int(k), n_known_null, a / loc_all.sum())
    for k, a in zip(by_location["null_rows"], by_location["all_rows"])
]
by_location = by_location.sort_values("null_rows", ascending=False)

print(f"{n_known_null} null rows have a known location, spread over {len(loc_null)} distinct "
      f"locations (of {len(loc_all):,} locations in the data).")
# tail_prob is not corrected for looking at many locations; the simulation below is the calibrated check.
by_location.head(10).assign(tail_prob=lambda d: d["tail_prob"].map("{:.1e}".format))

73 null rows have a known location, spread over 36 distinct locations (of 1,305 locations in the data).


,null_rows,all_rows,null_rate_%,expected_if_random,tail_prob
location,,,,,
Kanakpura Road,7,273,2.6,1.5,7.6e-04
Sarjapur Road,7,399,1.8,2.2,6.3e-03
Whitefield,6,540,1.1,3.0,7.6e-02
Devanahalli,5,45,11.1,0.2,5.5e-06
JP Nagar,5,72,6.9,0.4,5.1e-05
Hennur,3,55,5.5,0.3,3.5e-03
Yelahanka,3,213,1.4,1.2,1.1e-01
Jakkur,3,71,4.2,0.4,7.1e-03
KR Puram,3,91,3.3,0.5,1.4e-02


In [17]:
# Place the same number of null rows at random over all rows with a known location, 2,000 times.
# Per trial, record (1) how many land in the busiest location and (2) the smallest tail_prob over
# locations with at least MIN_ROWS null rows (one null row cannot form a cluster).
N_TRIALS, MIN_ROWS = 2000, 2

known = df["location"].notna()
codes, uniques = pd.factorize(df.loc[known, "location"])
listings = np.bincount(codes)
share = listings / len(codes)


def most_surprising(counts):
    """(smallest tail_prob, location index) over locations with >= MIN_ROWS null rows."""
    idx = np.flatnonzero(counts >= MIN_ROWS)
    if len(idx) == 0:
        return 1.0, None
    return min((tail_prob(int(counts[i]), n_known_null, share[i]), i) for i in idx)


obs_counts = np.bincount(codes[null_mask[known].to_numpy()], minlength=len(uniques))
obs_max = int(obs_counts.max())
obs_p, obs_j = most_surprising(obs_counts)

rng = np.random.default_rng(0)
sim_max = np.empty(N_TRIALS, dtype=int)
sim_p = np.empty(N_TRIALS)
for t in range(N_TRIALS):
    counts = np.bincount(codes[rng.choice(len(codes), size=n_known_null, replace=False)], minlength=len(uniques))
    sim_max[t] = counts.max()
    sim_p[t] = most_surprising(counts)[0]

p99 = int(np.percentile(sim_max, 99))
busiest = ", ".join(repr(name) for name in uniques[np.flatnonzero(obs_counts == obs_max)])
print(f"1) Does one location dominate? The busiest has {obs_max} of {n_known_null} null rows "
      f"({obs_max / n_known_null:.1%}; {busiest}). Random scatter: median {int(np.median(sim_max))}, "
      f"99th percentile {p99}. -> {'YES' if obs_max > p99 else 'no'}")
if obs_j is None:
    print(f"2) Is any location over-represented for its size? No location has {MIN_ROWS} or more null rows. -> no")
else:
    frac = (sim_p <= obs_p).mean()
    print(f"2) Is any location over-represented for its size? Most surprising: {uniques[obs_j]!r} "
          f"({obs_counts[obs_j]} null rows among {listings[obs_j]} listings). Random scatter produces a "
          f"location this surprising in {frac:.1%} of {N_TRIALS:,} trials. -> {'YES' if frac < 0.01 else 'no'}")
    print("   The same calibration for the three smallest tail_prob values (share of trials whose most surprising "
          "location is at least this surprising):")
    for loc in by_location[by_location["null_rows"] >= MIN_ROWS].sort_values("tail_prob").index[:3]:
        r = by_location.loc[loc]
        print(f"   {loc!r}: {int(r['null_rows'])} null rows among {int(r['all_rows'])} listings -> "
              f"{(sim_p <= r['tail_prob']).mean():.2%}")

1) Does one location dominate? The busiest has 7 of 73 null rows (9.6%; 'Kanakpura Road', 'Sarjapur  Road'). Random scatter: median 4, 99th percentile 8. -> no
2) Is any location over-represented for its size? Most surprising: 'Devanahalli' (5 null rows among 45 listings). Random scatter produces a location this surprising in 0.0% of 2,000 trials. -> YES
   The same calibration for the three smallest tail_prob values (share of trials whose most surprising location is at least this surprising):
   'Devanahalli': 5 null rows among 45 listings -> 0.00%
   'JP Nagar': 5 null rows among 72 listings -> 0.35%
   'Kanakpura Road': 7 null rows among 273 listings -> 13.70%


In [18]:
# Robustness: could spelling variants of one location name be hiding a cluster? Merge case and whitespace
# variants in a diagnostic copy only; the `location` column itself is left as it is (cleaning it comes later).
loc_norm = df["location"].str.split().str.join(" ").str.lower()
print(f"Distinct location names: {df['location'].nunique():,} as written, "
      f"{loc_norm.nunique():,} after merging case and whitespace variants.")
print("Busiest locations among the null rows after merging:", loc_norm[null_mask].value_counts().head(4).to_dict())

Distinct location names: 1,305 as written, 1,282 after merging case and whitespace variants.
Busiest locations among the null rows after merging: {'kanakpura road': 7, 'sarjapur road': 7, 'whitefield': 6, 'devanahalli': 5}


In [19]:
# The rows behind the three most over-represented locations (smallest tail_prob, at least MIN_ROWS null rows).
suspects = by_location[by_location["null_rows"] >= MIN_ROWS].sort_values("tail_prob").index[:3]
cols = ["area_type", "location", "size", "society", "total_sqft", "bath", "price"]
df_raw.loc[null_mask & df["location"].isin(suspects), cols].sort_values("location")

,area_type,location,size,society,total_sqft,bath,price
56,Built-up Area,Devanahalli,4 Bedroom,BrereAt,3010 - 3410,NaN,192.000
2862,Plot Area,Devanahalli,NaN,Ajleyor,1500 - 2400,NaN,46.800
5333,Plot Area,Devanahalli,NaN,Emngs S,2100 - 5405,NaN,177.115
224,Super built-up Area,Devanahalli,3 BHK,Jurdsig,1520 - 1740,NaN,74.820
13240,Super built-up Area,Devanahalli,1 BHK,Pardsri,1020 - 1130,NaN,52.570
669,Super built-up Area,JP Nagar,5 BHK,Pehtsa,4400 - 6640,NaN,375.000
702,Super built-up Area,JP Nagar,5 BHK,Pehtsa,4400 - 6800,NaN,548.500
801,Super built-up Area,JP Nagar,4 BHK,Pehtsa,4000 - 5249,NaN,453.000
6873,Super built-up Area,JP Nagar,3 BHK,Shestut,1750 - 2640,NaN,152.000
7398,Super built-up Area,JP Nagar,3 BHK,Pehtsa,3103 - 3890,NaN,237.500


In [20]:
# area_type is already dropped from df, but the raw frame still has it: diagnostic use only.
area_check = pd.DataFrame({
    "null_rows": df_raw.loc[null_mask, "area_type"].value_counts(),
    "all_rows": df_raw["area_type"].value_counts(),
}).fillna(0).astype(int)
area_check["null_rate_%"] = (area_check["null_rows"] / area_check["all_rows"] * 100).round(2)
exp_area = n_null_rows * area_check["all_rows"] / area_check["all_rows"].sum()  # expected if scattered at random
area_check["expected_if_random"] = exp_area.round(1)
area_check["lift"] = (area_check["null_rows"] / exp_area).round(2)
area_check.sort_values("null_rows", ascending=False)

,null_rows,all_rows,null_rate_%,expected_if_random,lift
area_type,,,,,
Super built-up Area,50,8790,0.57,48.8,1.02
Plot Area,16,2025,0.79,11.2,1.42
Built-up Area,8,2418,0.33,13.4,0.60
Carpet Area,0,87,0.00,0.5,0.00


In [21]:
# Where do the null rows sit in the file? Count them per tenth of the file (top -> bottom).
positions = np.flatnonzero(null_mask.to_numpy())
per_tenth = np.bincount(positions * 10 // len(df), minlength=10)
pd.DataFrame(
    [per_tenth],
    index=["null_rows"],
    columns=[f"{i * 10}-{(i + 1) * 10}%" for i in range(10)],
)

,0-10%,10-20%,20-30%,30-40%,40-50%,50-60%,60-70%,70-80%,80-90%,90-100%
null_rows,12,8,8,5,10,8,10,5,5,3


### Findings: where the null rows sit

- **74 distinct rows are affected, not the 90 you get by adding the three column counts.** The nulls overlap: the 16 `size` nulls are all also `bath` nulls, 57 rows are null in `bath` only, and 1 row is null in `location` only.
- **Not one location.** 73 of the rows have a known location, spread over 36 of the 1,305 locations. The busiest, Kanakpura Road and Sarjapur Road, hold 7 each (9.6%), within what random scatter produces (median 4, 99th percentile 8). Merging case and whitespace variants of the names (1,305 become 1,282) leaves the busiest counts unchanged.
- **Two small locations stand out for their size.** Devanahalli (5 null rows among 45 listings) and JP Nagar (5 among 72) each hold more null rows than random scatter produces, even after allowing for having looked at every location; Kanakpura Road (7 among 273), the busiest by count, does not (calibration printed above). The two look like listing type rather than place: all 5 of Devanahalli's and all 5 of JP Nagar's null rows have a ranged `total_sqft`. Kanakpura Road's 7 come from just two societies (5 and 2 rows).
- **The clearest pattern is listing type, not place.** The 16 `size` nulls are all Plot Area listings with availability "Immediate Possession". Of the 57 `bath`-only nulls, 49 (86%) have `total_sqft` written as a range, against 1.5% of all rows, and every `bath` null also lacks `balcony`.
- **By `area_type`** the null rate is 0.3% to 0.8% in each of the three large types (0 of 87 in Carpet Area). **By position**, the rows are spread over the whole file (3 to 12 per tenth).

**Decision: drop these rows rather than impute them.** `location` has no sensible fill value, and `bath` is structurally tied to the bedroom count in `size`, so a median fill would invent data. The affected share is 0.56% of the rows. The drop also removes 57 of the 201 raw rows with a ranged `total_sqft`, so 144 remain for Session 2.

In [22]:
rows_before = len(df)
df = df.dropna()  # the index is deliberately not reset: it keeps each row traceable to its line in the raw CSV
rows_after = len(df)
assert rows_before - rows_after == n_null_rows  # exactly the rows flagged above

log_step(
    "Drop rows with a null in `location`, `size` or `bath`",
    "`location` has no sensible fill value; `bath` is tied to bedroom count (`size`), so imputing would invent data",
    rows_before, rows_after,
)

print(f"Rows before: {rows_before:,} | after: {rows_after:,} | removed: {rows_before - rows_after:,} "
      f"({(rows_before - rows_after) / rows_before:.2%})")

Rows before: 13,320 | after: 13,246 | removed: 74 (0.56%)


## 5. Final check

In [23]:
assert list(df.columns) == ["location", "size", "total_sqft", "bath", "price"]
assert df.isnull().sum().sum() == 0

print(f"Final shape: {df.shape}")

null_summary = pd.DataFrame({"nulls_before": df_raw.isnull().sum()})
null_summary["nulls_after"] = df.isnull().sum()  # aligns on column name; dropped columns become missing
null_summary.astype("Int64")

Final shape: (13246, 5)


,nulls_before,nulls_after
area_type,0,<NA>
availability,0,<NA>
location,1,0
size,16,0
society,5502,<NA>
total_sqft,0,0
bath,73,0
balcony,609,<NA>
price,0,0


`<NA>` in `nulls_after` marks a column that was dropped.

In [24]:
df.head()

,location,size,total_sqft,bath,price
0,Electronic City Phase II,2 BHK,1056,2.0,39.07
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00
2,Uttarahalli,3 BHK,1440,2.0,62.00
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00
4,Kothanur,2 BHK,1200,2.0,51.00


In [25]:
# Guard: the written findings and the decision log below describe one specific file and its row counts.
# Say so if a run no longer matches them.
WRITTEN_FOR_SHA256 = "23ddf71be7d7"  # first 12 characters of the file's SHA-256
written_log = [(13320, 13320)] * 4 + [(13320, 13246)]  # (rows before, rows after) per decision
recorded_log = [(s["rows_before"], s["rows_after"]) for s in decision_log]
same_file = csv_sha256.startswith(WRITTEN_FOR_SHA256)
if same_file and recorded_log == written_log:
    print("The written findings and decision log match this run.")
else:
    print(f"WARNING: this run differs from the data the written findings describe "
          f"(same file: {same_file}, same row counts: {recorded_log == written_log}); update them.")

The written findings and decision log match this run.


## 6. Decision log

Each decision with its row impact, as recorded while the notebook ran on this data.

| # | Decision | Why | Rows before | Rows after | Rows removed |
|--:|----------|-----|------------:|-----------:|-------------:|
| 1 | Drop `area_type` | Judgment call: changes what `total_sqft` measures; re-test in Session 5 | 13,320 | 13,320 | 0 |
| 2 | Drop `availability` | Possession label/month, not a property attribute | 13,320 | 13,320 | 0 |
| 3 | Drop `society` | Large share missing and very high-cardinality | 13,320 | 13,320 | 0 |
| 4 | Drop `balcony` | Minor amenity count; avoids imputing | 13,320 | 13,320 | 0 |
| 5 | Drop rows with a null in `location`, `size` or `bath` | `location` has no sensible fill value; `bath` is tied to bedroom count (`size`), so imputing would invent data | 13,320 | 13,246 | 74 |

**Result:** 13,320 rows × 9 columns → 13,246 rows × 5 columns (`location`, `size`, `total_sqft`, `bath`, `price`), zero nulls. 74 rows removed, 0.56% of the raw data (the brief expected under 1%). Prices are still in lakhs of INR.